# Phase 11 — Error Handling & Circuit Breaker

> **Gemini-only project.** Every cell here uses Google Gemini (chat + embeddings) via `langchain-google-genai`. The only key the core needs is `GOOGLE_API_KEY`. This notebook is a **scaffold** — run it top-to-bottom *after* Claude Code finishes this phase. If a cell references a module that doesn't exist yet, that phase hasn't been built.

**Goal:** a failing adapter opens its circuit; yfinance fallback keeps the user unaffected.


In [ ]:
# --- Setup: load .env and make the project importable ---
import sys, os
sys.path.insert(0, os.path.abspath('..'))  # repo root
from dotenv import load_dotenv
load_dotenv('../.env', override=True)  # override=True: beat VS Code's stale cached env vars

# This project is Gemini-only. Confirm the one required key is present.
assert os.getenv('GOOGLE_API_KEY'), 'Set GOOGLE_API_KEY in .env (see 00_prerequisites_and_setup.md)'
print('GOOGLE_API_KEY loaded:', bool(os.getenv('GOOGLE_API_KEY')))
print('SEC_USER_AGENT   :', os.getenv('SEC_USER_AGENT', '(set before Phase 5)'))

## 1. Circuit opens after N consecutive failures

In [ ]:
from app.errors.circuit_breaker import CircuitBreaker, CircuitOpenError
cb = CircuitBreaker(threshold=3, cool_off=60)
def always_fails():
    raise RuntimeError('adapter down')
opened_on = None
for i in range(1, 6):
    try:
        cb.call(always_fails)
    except CircuitOpenError:
        opened_on = opened_on or i; print(f'attempt {i}: circuit OPEN (fail fast)')
    except RuntimeError:
        print(f'attempt {i}: real call failed')
print('circuit opened starting at attempt', opened_on)

## 2. Fallback keeps the answer flowing

In [ ]:
from app.errors.fallback import Fallback
res = Fallback(primary=always_fails, secondary=lambda: 'yfinance result').run()
print('fallback returned:', res)

## ✅ Acceptance check

In [ ]:
assert opened_on == 4 or opened_on == 3  # opens on/after the 3rd failure
print('Phase 11 acceptance: PASS')